# Publication notebook: external-test assembly inspection

- Purpose: inspect external-test assembly outputs and derived graph metrics.
- Required inputs: local `data/assemblies_spades/final_run/` assembly directories.
- Required models: none at notebook runtime.
- Required external tools: prior assembly outputs must already exist locally.
- Expected outputs: external-test summary TSVs under `results/tables/`.
- Publication output: supplementary assembly-inspection tables.
- Reproduction status: normalized for portability, but remains local-only because the required assembly outputs are not tracked.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
ASSEMBLIES_ROOT = PROJECT_ROOT / "data" / "assemblies_spades" / "final_run"

for p in sorted(ASSEMBLIES_ROOT.iterdir()):
    if p.is_dir():
        print(p.name)
        for child in sorted(p.iterdir())[:10]:
            print("   ", child.name)
        print()

In [ ]:
from pathlib import Path
import re
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()
ASSEMBLIES_ROOT = PROJECT_ROOT / "data" / "assemblies_spades" / "final_run"

print("PROJECT_ROOT     =", PROJECT_ROOT)
print("ASSEMBLIES_ROOT  =", ASSEMBLIES_ROOT)

def read_text(p: Path) -> str:
    try:
        return p.read_text(errors="ignore")
    except Exception:
        return ""

def fasta_contig_lengths(fa_path: Path):
    lens = []
    cur = 0
    for line in fa_path.read_text(errors="ignore").splitlines():
        if line.startswith(">"):
            if cur > 0:
                lens.append(cur)
                cur = 0
        else:
            cur += len(line.strip())
    if cur > 0:
        lens.append(cur)
    return lens

def n50_l50(lengths):
    if not lengths:
        return None, None
    lengths = sorted(lengths, reverse=True)
    total = sum(lengths)
    half = total / 2
    csum = 0
    for i, L in enumerate(lengths, start=1):
        csum += L
        if csum >= half:
            return L, i
    return lengths[-1], len(lengths)

def gfa_stats(gfa_path: Path):
    nodes = 0
    edges = 0
    for line in gfa_path.read_text(errors="ignore").splitlines():
        if line.startswith("S\t"):
            nodes += 1
        elif line.startswith("L\t"):
            edges += 1
    return nodes, edges

def find_gfa(run_root: Path):
    for name in [
        "assembly_graph_with_scaffolds.gfa",
        "assembly_graph.gfa",
        "assembly_graph_with_scaffolds.gfa.gz",
        "assembly_graph.gfa.gz",
    ]:
        p = run_root / name
        if p.exists():
            return p
    return None

def parse_spades_log(run_root: Path):
    candidates = [
        run_root / "spades.log",
        run_root / "spades.log.txt",
        run_root / "params.txt",
    ]
    txt = ""
    for c in candidates:
        if c.exists():
            txt += "\n" + read_text(c)

    reads_pairs = None
    patterns = [
        r"Number of read-?pairs:\s*(\d+)",
        r"Total\s+pairs:\s*(\d+)",
        r"total\s+paired\s+reads:\s*(\d+)",
        r"paired\s+reads:\s*(\d+)",
    ]
    for pat in patterns:
        m = re.search(pat, txt, flags=re.I)
        if m:
            reads_pairs = int(m.group(1))
            break

    return {"reads_pairs_used": reads_pairs}

def summarize_spades_run(run_root: Path):
    contig_hits = list(run_root.rglob("contigs.fasta"))
    scaffold_hits = list(run_root.rglob("scaffolds.fasta"))

    if not contig_hits:
        return None

    contigs = contig_hits[0]
    scaffolds = scaffold_hits[0] if scaffold_hits else None

    lens = fasta_contig_lengths(contigs)
    n_contigs = len(lens)
    total_bp = sum(lens) if lens else 0
    max_len = max(lens) if lens else None
    n50, l50 = n50_l50(lens)

    scaff_lens = fasta_contig_lengths(scaffolds) if scaffolds else []
    n_scaff = len(scaff_lens) if scaff_lens else None
    scaff_bp = sum(scaff_lens) if scaff_lens else None
    scaff_n50, scaff_l50 = n50_l50(scaff_lens) if scaff_lens else (None, None)

    gfa = find_gfa(contigs.parent)
    nodes = edges = None
    if gfa and gfa.suffix == ".gz":
        nodes = edges = None
    elif gfa:
        nodes, edges = gfa_stats(gfa)

    logm = parse_spades_log(contigs.parent)

    return {
        "run_root": str(run_root),
        **logm,
        "contigs_fasta": str(contigs),
        "contigs_n": n_contigs,
        "contigs_bp": total_bp,
        "contigs_max": max_len,
        "contigs_N50": n50,
        "contigs_L50": l50,
        "scaffolds_fasta": str(scaffolds) if scaffolds else None,
        "scaffolds_n": n_scaff,
        "scaffolds_bp": scaff_bp,
        "scaffolds_N50": scaff_n50,
        "scaffolds_L50": scaff_l50,
        "graph_gfa": str(gfa) if gfa else None,
        "gfa_nodes": nodes,
        "gfa_edges": edges,
    }
def parse_folder_name(folder: str):
    """
    Handles both naming schemes:

    Old:
      10K_final_10_unfiltered
      10K_final_10_gb_t0p5
      10K_final_10_cnn_fixedep25_t0p5

    New BiGRU:
      bigru_10k_10_bigru_filtered
      bigru_10k_10_unfiltered
      bigru_3k_5_bigru_filtered
      bigru_unfiltered   # ignore this if no dataset suffix
      bigru_bigru_filtered  # ignore this if no dataset suffix
    """

    # -------------------------
    # Old naming scheme
    # -------------------------
    m = re.match(r"^(3200|10K|20K)_final_(\d+)_(.+)$", folder, flags=re.I)
    if m:
        read_level = m.group(1).upper()
        chim = m.group(2)
        suffix = m.group(3).lower()

        dataset = f"{read_level}_final_{chim}"

        if suffix == "unfiltered":
            return dataset, "unfiltered"
        if suffix.startswith("gb"):
            return dataset, "gb"
        if suffix.startswith("cnn"):
            return dataset, "cnn"

        return None, None

    # -------------------------
    # New BiGRU naming scheme
    # -------------------------
    m = re.match(r"^bigru_(10k|20k|3k)_(\d+)_(bigru_filtered|unfiltered)$", folder, flags=re.I)
    if m:
        rl_raw = m.group(1).lower()
        chim = m.group(2)
        suffix = m.group(3).lower()

        rl_map = {
            "3k": "3200",
            "10k": "10K",
            "20k": "20K",
        }
        read_level = rl_map[rl_raw]
        dataset = f"{read_level}_final_{chim}"

        if suffix == "unfiltered":
            return dataset, "unfiltered"
        if suffix == "bigru_filtered":
            return dataset, "bigru"

    return None, None
rows = []

if not ASSEMBLIES_ROOT.exists():
    raise FileNotFoundError(f"Missing folder: {ASSEMBLIES_ROOT}")

for run_root in sorted([p for p in ASSEMBLIES_ROOT.iterdir() if p.is_dir()]):
    dataset, flt = parse_folder_name(run_root.name)
    if dataset is None:
        continue

    rec = summarize_spades_run(run_root)

    if rec is None:
        rows.append({
            "dataset": dataset,
            "filter": flt,
            "status": "MISSING",
            "run_root": str(run_root),
        })
    else:
        rows.append({
            "dataset": dataset,
            "filter": flt,
            "status": "OK",
            **rec,
        })

df = pd.DataFrame(rows).sort_values(["dataset", "filter"]).reset_index(drop=True)

missing = df[df["status"] != "OK"][["dataset", "filter", "run_root"]]
if len(missing):
    print("Missing runs (folder exists but contigs.fasta not found):")
    display(missing)

df

In [ ]:
df[["dataset", "filter"]].value_counts().sort_index()

In [ ]:
df_ok = df[df["status"] == "OK"].copy()

# Prefer rows with more complete assembly information if duplicates exist
df_ok = (
    df_ok.sort_values(
        ["dataset", "filter", "contigs_bp", "contigs_n"],
        ascending=[True, True, False, False]
    )
    .drop_duplicates(subset=["dataset", "filter"], keep="first")
    .reset_index(drop=True)
)

df_ok[["dataset", "filter", "contigs_n", "contigs_bp", "run_root"]]

In [ ]:
unf = df_ok[df_ok["filter"] == "unfiltered"].copy()
flt = df_ok[df_ok["filter"] != "unfiltered"].copy()

merged = flt.merge(
    unf[[
        "dataset",
        "reads_pairs_used",
        "contigs_n",
        "contigs_bp",
        "contigs_N50",
        "contigs_max",
        "gfa_nodes",
        "gfa_edges",
    ]],
    on="dataset",
    how="left",
    suffixes=("", "_unf"),
)

num_cols = [
    "reads_pairs_used", "reads_pairs_used_unf",
    "contigs_n", "contigs_n_unf",
    "contigs_bp", "contigs_bp_unf",
    "contigs_N50", "contigs_N50_unf",
    "contigs_max", "contigs_max_unf",
    "gfa_nodes", "gfa_nodes_unf",
    "gfa_edges", "gfa_edges_unf",
]
for c in num_cols:
    if c in merged.columns:
        merged[c] = pd.to_numeric(merged[c], errors="coerce")

denom = merged["reads_pairs_used_unf"]
numer = merged["reads_pairs_used"]
reads_kept = (numer / denom) * 100
merged["reads_kept_pct"] = reads_kept.where(denom.notna() & (denom != 0) & numer.notna())
merged["reads_kept_pct"] = merged["reads_kept_pct"].round(2)

merged["delta_bp"] = merged["contigs_bp"] - merged["contigs_bp_unf"]
merged["delta_N50"] = merged["contigs_N50"] - merged["contigs_N50_unf"]
merged["delta_max"] = merged["contigs_max"] - merged["contigs_max_unf"]

results = (
    merged[[
        "dataset", "filter",
        "reads_pairs_used_unf", "reads_pairs_used", "reads_kept_pct",
        "contigs_n_unf", "contigs_n",
        "contigs_bp_unf", "contigs_bp", "delta_bp",
        "contigs_N50_unf", "contigs_N50", "delta_N50",
        "contigs_max_unf", "contigs_max", "delta_max",
        "gfa_nodes_unf", "gfa_nodes",
        "gfa_edges_unf", "gfa_edges",
        "run_root",
    ]]
    .rename(columns={
        "reads_pairs_used_unf": "reads_pairs_unfiltered",
        "reads_pairs_used": "reads_pairs_filtered",
        "contigs_n_unf": "contigs_unfiltered",
        "contigs_n": "contigs_filtered",
        "contigs_bp_unf": "bp_unfiltered",
        "contigs_bp": "bp_filtered",
        "contigs_N50_unf": "N50_unfiltered",
        "contigs_N50": "N50_filtered",
        "contigs_max_unf": "max_unfiltered",
        "contigs_max": "max_filtered",
        "gfa_nodes_unf": "nodes_unfiltered",
        "gfa_nodes": "nodes_filtered",
        "gfa_edges_unf": "edges_unfiltered",
        "gfa_edges": "edges_filtered",
    })
    .sort_values(["dataset", "filter"])
    .reset_index(drop=True)
)

results

In [ ]:
out_path = PROJECT_ROOT / "results" / "tables" / "external_test_spades_results.tsv"
results.to_csv(out_path, sep="\t", index=False)
print("Wrote:", out_path)

In [ ]:
results.columns.tolist()

In [ ]:
results["filter"].value_counts(dropna=False)

In [ ]:
expected = []
for rl in ["3200", "10K", "20K"]:
    for chim in [5, 10, 15, 50]:
        expected.append(f"{rl}_final_{chim}")

bigru_seen = set(df_ok.loc[df_ok["filter"] == "bigru", "dataset"])
missing_bigru = sorted(set(expected) - bigru_seen)

print("Missing BIGRU datasets:")
print(missing_bigru)

## debug

In [ ]:
from pathlib import Path

run_root = Path("../data/assemblies_spades/final_run/10K_final_5_gb_t0p5")
contig_hits = list(run_root.rglob("contigs.fasta"))
print("contig_hits:", contig_hits)

if contig_hits:
    contigs = contig_hits[0]
    print("contigs:", contigs)
    print("contigs.parent:", contigs.parent)

    for name in ["spades.log", "spades.log.txt", "params.txt"]:
        p = contigs.parent / name
        print(name, p.exists(), p)

In [ ]:
log_path = contigs.parent / "spades.log"
if log_path.exists():
    txt = log_path.read_text(errors="ignore")
    print(txt[:5000])
else:
    print("spades.log not found")

In [ ]:
params_path = contigs.parent / "params.txt"
if params_path.exists():
    txt = params_path.read_text(errors="ignore")
    print(txt[:3000])
else:
    print("params.txt not found")